# GridLeak AI — Ön İşleme ve Özellikler

Mevcut master defterin hücreleri içerikleri ve kayıtlı çıktıları korunarak bu deftere taşınmıştır. Ortak hazırlık hücreleri bağımsız açılış için tekrar yer alır.

Bu düzenleme yeni eğitim veya değerlendirme çalıştırmamıştır. Hücreleri çalıştırmak ilgili hesaplamaları yeniden yürütür.

# GridLeak AI (v3) — Uçtan Uca Açıklamalı Master Defter

Bu defter, projedeki gerçek üretim modüllerini (**birebir aynı kodları**) doğrudan içe aktararak tüm makine öğrenmesi akışını adım adım çalıştırır ve görselleştirir:

- **Özellik Mühendisliği:** `ai/features/v3_features.py` (581 aday → 160 seçilmiş özellik)
- **Model Mimarisi ve Eğitim:** `ai/training/v3_final_train_and_test.py` (LightGBM + CatBoost + XGBoost + RankGauss MLP + Sigmoid Stacking)
- **Değerlendirme, TreeSHAP ve Katalog Aktarımı:** `ai/evaluation/v3_export_and_shap.py` & `export_application.py`

> **Veri Seti Bağlantısı (SGCC):** https://www.kaggle.com/datasets/bensalem14/sgcc-dataset

In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, average_precision_score, roc_auc_score, f1_score,
    precision_score, recall_score, precision_recall_curve, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay
)
warnings.filterwarnings('ignore')

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ai/features/v3_features.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Projedeki GERÇEK üretim modüllerini doğrudan içe aktarıyoruz (sıfır kod tekrarı!)
from ai.features.v3_features import load_raw_for_ids, extract_subscriber_features, FoldContextTransformer
from ai.training.v3_final_train_and_test import build_base_estimators, build_meta_estimator, ece_score

EXP = ROOT / 'ai/artifacts/iyilestirme_v3_20261123T171349Z'
V3 = ROOT / 'data/processed/v3'
lock = json.loads((EXP / 'locked_selection.json').read_text(encoding='utf-8'))
fcols = lock['features']
print('Proje Kökü:', ROOT)
print('Aktif Kilitli Model:', lock['model_name'])
print('Kilitli Özellik Sayısı:', len(fcols))

Proje Kökü: c:\Users\kadir\GridLeakAI
Aktif Kilitli Model: Calibrated Stacking v3 (LGBM + CatBoost + XGBoost + MLP)
Kilitli Özellik Sayısı: 160


## 1. Veri Seti Yapısı, %80/%20 Abone Bölünmesi ve Sınıf Dengesizliği
Aboneler kimlik (`CONS_NO`) bazında tabakalı bölünmüştür; eğitim ve test arasında **sıfır ortak abone** vardır.

In [2]:
train_df = pd.read_csv(V3 / 'egitim_ozellikleri_2016_v1.csv', dtype={'CONS_NO': str})
test_df = pd.read_csv(V3 / 'test_ozellikleri_2016_v1.csv', dtype={'CONS_NO': str})
test_y_df = pd.read_csv(V3 / 'test_hedefleri_2016_v1.csv', dtype={'CONS_NO': str}).set_index('CONS_NO')
assert not (set(train_df['CONS_NO']) & set(test_df['CONS_NO'])), 'Eğitim/Test sızıntısı!'

y_tr = train_df['FLAG'].to_numpy(int)
y_te = test_y_df.loc[test_df['CONS_NO'], 'FLAG'].to_numpy(int)
print(f'Eğitim: {len(train_df):,} abone | FLAG=0: {(y_tr==0).sum():,} (%{(y_tr==0).mean()*100:.2f}) | FLAG=1: {(y_tr==1).sum():,} (%{y_tr.mean()*100:.2f})')
print(f'Test  : {len(test_df):,} abone | FLAG=0: {(y_te==0).sum():,} (%{(y_te==0).mean()*100:.2f}) | FLAG=1: {(y_te==1).sum():,} (%{y_te.mean()*100:.2f})')
print('Eğitim-Test Ortak Abone Sayısı: 0')

Eğitim: 33,858 abone | FLAG=0: 30,996 (%91.55) | FLAG=1: 2,862 (%8.45)
Test  : 8,466 abone | FLAG=0: 7,748 (%91.52) | FLAG=1: 718 (%8.48)
Eğitim-Test Ortak Abone Sayısı: 0


## 2. Canlı Özellik Çıkarımı Doğrulaması (`ai/features/v3_features.py`)
Aşağıdaki hücre, dışarıdaki `v3_features.py` motorunu ve yalnızca eğitime fit edilmiş `context_transformer.joblib` dönüştürücüsünü **canlı olarak** 5 test abonesi üzerinde çalıştırıp ham 1.034 günlük tüketimden **581 aday özellik → 160 kilitli özellik** üretimini doğrular.

In [3]:
ctx_fitted = joblib.load(EXP / 'context_transformer.joblib')
v2_te = pd.read_csv(ROOT / 'data/processed/v2/test_ozellikleri_2016_v1.csv', dtype={'CONS_NO': str})
sample_ids = v2_te['CONS_NO'].head(5).tolist()
base49_sample = v2_te.head(5).drop(columns=['CONS_NO']).set_index(pd.Index(sample_ids, name='CONS_NO'))

arr_s, dates_s = load_raw_for_ids(sample_ids)
sub_s = extract_subscriber_features(arr_s, dates_s, base49_sample, include_history=True)
full_s = pd.concat([sub_s, ctx_fitted.transform(arr_s, sub_s)], axis=1)
selected_s = full_s[fcols]
print(f'Ham matris: {arr_s.shape} (5 abone × 1.034 gün) -> Aday özellik: {full_s.shape[1]} -> Seçilmiş: {selected_s.shape[1]}')
display(selected_s.iloc[:, :8].round(3))

Ham matris: (5, 1034) (5 abone × 1.034 gün) -> Aday özellik: 581 -> Seçilmiş: 160


,h_missing_episodes_after_start,h_all_std,h_step_jump_ratio_all,h_2014_missing_episodes,h_missing_mean_gap_len,ctx_svd_mafter_4,h_2014_autocorr_1,h_oct2016_vs_oct2014
CONS_NO,,,,,,,,
937E769D8B0DFB752CB8CA5F66050A51,2.0,7.357,2.840,2.0,113.50,-2.807,0.725,2.204
28FA25303235F58B57A7E78963030522,1.0,0.000,1.000,1.0,1.00,0.006,NaN,NaN
435025C982D7CAB582DE393467113EAB,2.0,3.456,1.920,1.0,4.00,0.090,NaN,NaN
F9A7E10C5CDA7F92F4C77A572ACB1093,4.0,3.594,1.789,2.0,23.25,1.754,0.880,1.468
142C934EF003BD1735EE448163CAE929,0.0,4.708,1.000,1.0,0.00,0.000,NaN,NaN
